# Mission 02: MongoDB Operations Desk

A support platform stores tickets, users and message histories as **documents**. You run the operations desk: design the document model, implement the ticket workflow, and answer operational questions with aggregations.

**Lessons used:** 12 (MongoDB), 10 (indexes), 13 (Redis, optional bonus).

The tickets are **synthetic** (generated below); they contain no real people.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from pymongo import ASCENDING, MongoClient

sys.path.insert(0, str(Path.cwd().parent.parent / "tools"))
import datasets

datasets.ensure_service("db", "mongodb")
client = MongoClient("mongodb://localhost:27017", serverSelectionTimeoutMS=20_000)
db = client["support_desk"]

import random
from datetime import datetime, timedelta

def make_tickets(n: int = 2000, seed: int = 7) -> list[dict]:
    """Synthetic support tickets (no real people): users, priorities, and a message history per ticket."""
    rnd = random.Random(seed)
    start = datetime(2025, 1, 1)
    tickets = []
    for i in range(1, n + 1):
        opened = start + timedelta(minutes=rnd.randint(0, 365 * 24 * 60))
        priority = rnd.choices(["low", "medium", "high", "urgent"], [40, 35, 20, 5])[0]
        status = rnd.choices(["open", "pending", "closed"], [25, 15, 60])[0]
        messages = [{"author": "customer", "at": opened, "text": "Problem report"}]
        if status != "open" or rnd.random() < 0.5:
            speed = {"urgent": 30, "high": 120, "medium": 480, "low": 1440}[priority]
            first_reply = opened + timedelta(minutes=rnd.expovariate(1 / speed))
            messages.append({"author": "agent", "at": first_reply, "text": "We are looking into it"})
            for k in range(rnd.randint(0, 4)):
                messages.append({"author": rnd.choice(["customer", "agent"]),
                                 "at": messages[-1]["at"] + timedelta(minutes=rnd.randint(5, 600)), "text": f"follow-up {k}"})
        tickets.append({"ticket_id": i, "user": {"id": rnd.randint(1, 300), "region": rnd.choice(["EU", "US", "APAC"]),
                                                 "plan": rnd.choice(["free", "pro", "enterprise"])},
                        "subject": rnd.choice(["login", "billing", "bug", "feature request", "performance"]),
                        "priority": priority, "status": status, "opened_at": opened, "messages": messages})
    return tickets

tickets = make_tickets()
tickets[0]

## Tasks

1. **Document design.** Write `create_ticket_document(ticket)`: validate the required fields and allowed values, and return the document to insert. Explain in 3–5 sentences why the messages are embedded and the users are not a separate collection (or why you chose otherwise).
2. **Create.** Insert all tickets into the collection `tickets`.
3. **Read.** Count the open `high` or `urgent` tickets of EU users on the `enterprise` plan (nested fields).
4. **Update.** Assign an agent to all open urgent tickets (`$set`) and add an agent message (`$push`).
5. **Delete.** Archive (copy to `tickets_archive`) and delete closed tickets opened before 1 April 2025.
6. **Aggregate.** Write `summarize_tickets(collection)`: open tickets per region and priority. Then compute the **median time to first agent reply** per priority, and plot it.
7. **Index.** Create an index for the query in task 3 and show with `explain()` that MongoDB uses it.

In [ ]:
ALLOWED_PRIORITIES = {"low", "medium", "high", "urgent"}
ALLOWED_STATUS = {"open", "pending", "closed"}

def create_ticket_document(ticket: dict) -> dict:
    """Validate a ticket and return the document to insert into MongoDB."""
    ...

def summarize_tickets(collection) -> pd.DataFrame:
    """Return the number of open tickets per region and priority."""
    ...

In [ ]:
# Tasks 2-7
...

## Submission Checklist

- [ ] Invalid tickets are rejected with a clear message.
- [ ] Create, read, update and delete are demonstrated.
- [ ] At least one query on a nested field and one on an array.
- [ ] An aggregation answers an operational question, with a plot.
- [ ] `explain()` shows an index scan (`IXSCAN`).

A solution is in `02-MongoDB-Operations-Desk-solution.ipynb`. Try the mission first.